In [4]:
import simpy
import numpy as np
import pandas as pd

In [ ]:
# Problema 1

In [5]:
def tiempo_cajero_1():
    t = np.random.normal(3.5, 0.8)
    while t <= 0: 
        t = np.random.normal(3.5, 0.8)
    return t

def tiempo_cajero_2():
    return np.random.uniform(4, 7)

In [ ]:
def cliente_escenario_a(env, cajero1, cajero2, stats):
    t_llegada = env.now
    
    if cajero1.count == 0:
        req = cajero1.request()
        cajero_elegido = 1
        yield req
    elif cajero2.count == 0:
        req = cajero2.request()
        cajero_elegido = 2
        yield req
    else:
        req1 = cajero1.request()
        req2 = cajero2.request()
        
        resultado = yield env.any_of([req1, req2])
        
        if req1 in resultado:
            req = req1
            req2.cancel()
            cajero_elegido = 1
        else:
            req = req2
            req1.cancel()
            cajero_elegido = 2

    t_espera = env.now - t_llegada
    stats['tiempos_espera'].append(t_espera)
    
    if cajero_elegido == 1:
        t_servicio = tiempo_cajero_1()
        yield env.timeout(t_servicio)
        cajero1.release(req)
        stats['uso_cajero1'] += t_servicio
    else:
        t_servicio = tiempo_cajero_2()
        yield env.timeout(t_servicio)
        cajero2.release(req)
        stats['uso_cajero2'] += t_servicio
        
    stats['tiempos_sistema'].append(t_espera + t_servicio)

def generador_llegadas_a(env, cajero1, cajero2, stats, tiempo_limite):
    while env.now < tiempo_limite:
        yield env.timeout(np.random.exponential(4))
        env.process(cliente_escenario_a(env, cajero1, cajero2, stats))

In [7]:
def cliente_escenario_b(env, cajero, tipo_cajero, stats):
    t_llegada = env.now
    req = cajero.request()
    yield req
    
    t_espera = env.now - t_llegada
    stats['tiempos_espera'].append(t_espera)
    
    if tipo_cajero == 1:
        t_servicio = tiempo_cajero_1()
        yield env.timeout(t_servicio)
        stats['uso_cajero1'] += t_servicio
    else:
        t_servicio = tiempo_cajero_2()
        yield env.timeout(t_servicio)
        stats['uso_cajero2'] += t_servicio
        
    cajero.release(req)
    stats['tiempos_sistema'].append(t_espera + t_servicio)

def generador_llegadas_b(env, cajero, tipo_cajero, stats, tiempo_limite):
    while env.now < tiempo_limite:
        yield env.timeout(np.random.exponential(8))
        env.process(cliente_escenario_b(env, cajero, tipo_cajero, stats))

In [ ]:
def simular_turnos(escenario, num_turnos, minutos_por_turno=240):
    resultados = {'espera': [], 'sistema': [], 'util_c1': [], 'util_c2': []}
    
    for _ in range(num_turnos):
        env = simpy.Environment()
        cajero1 = simpy.Resource(env, capacity=1)
        cajero2 = simpy.Resource(env, capacity=1)
        stats = {'tiempos_espera': [], 'tiempos_sistema': [], 'uso_cajero1': 0, 'uso_cajero2': 0}
        
        if escenario == 'A':
            env.process(generador_llegadas_a(env, cajero1, cajero2, stats, minutos_por_turno))
        else:
            env.process(generador_llegadas_b(env, cajero1, 1, stats, minutos_por_turno))
            env.process(generador_llegadas_b(env, cajero2, 2, stats, minutos_por_turno))
            
        env.run()
    
        if stats['tiempos_espera']:
            resultados['espera'].append(np.mean(stats['tiempos_espera']))
            resultados['sistema'].append(np.mean(stats['tiempos_sistema']))
        resultados['util_c1'].append((stats['uso_cajero1'] / minutos_por_turno) * 100)
        resultados['util_c2'].append((stats['uso_cajero2'] / minutos_por_turno) * 100)
        
    return {
        'Espera Promedio (min)': round(np.mean(resultados['espera']), 2),
        'Sistema Promedio (min)': round(np.mean(resultados['sistema']), 2),
        'Utilización Cajero 1 (%)': round(np.mean(resultados['util_c1']), 2),
        'Utilización Cajero 2 (%)': round(np.mean(resultados['util_c2']), 2)
    }

turnos_a_simular = [1, 5, 10, 30]

for turnos in turnos_a_simular:
    print(f"\n{'='*50}")
    print(f"RESULTADOS SIMULANDO {turnos} TURNOS DE 240 MINUTOS")
    print(f"{'='*50}")
    
    res_A = simular_turnos('A', turnos)
    res_B = simular_turnos('B', turnos)
    
    df = pd.DataFrame([res_A, res_B], index=['Escenario A (Cola Única)', 'Escenario B (Colas Separadas)'])
    print(df.to_string())


RESULTADOS SIMULANDO 1 TURNO(S) DE 240 MINUTOS
                               T. Espera Promedio (min)  T. Sistema Promedio (min)  Utilización Cajero 1 (%)  Utilización Cajero 2 (%)
Escenario A (Cola Única)                           0.47                       4.49                     52.21                     33.28
Escenario B (Colas Separadas)                      2.73                       6.93                     47.86                     50.20

RESULTADOS SIMULANDO 5 TURNO(S) DE 240 MINUTOS
                               T. Espera Promedio (min)  T. Sistema Promedio (min)  Utilización Cajero 1 (%)  Utilización Cajero 2 (%)
Escenario A (Cola Única)                           0.86                       5.02                     60.34                     49.73
Escenario B (Colas Separadas)                      2.11                       6.64                     41.27                     67.96

RESULTADOS SIMULANDO 10 TURNO(S) DE 240 MINUTOS
                               T. Espera Prom

In [9]:
#Problema 2
def tiempo_emp1_A():
    t = np.random.normal(4, 1)
    while t <= 0:
        t = np.random.normal(4, 1)
    return t

def tiempo_emp2_A():
    return np.random.uniform(5, 8)

def tiempo_kiosco_B():
    return np.random.uniform(2, 4)

def tiempo_entrega_B():
    t = np.random.normal(3, 0.5)
    while t <= 0:
        t = np.random.normal(3, 0.5)
    return t

def tiempo_tradicional_B():
    return np.random.uniform(5, 8)

In [13]:
def cliente_escenario_a(env, emp1, emp2, stats):
    t_llegada = env.now
    
    if emp1.count == 0:
        req = emp1.request()
        emp_elegido = 1
        yield req
    elif emp2.count == 0:
        req = emp2.request()
        emp_elegido = 2
        yield req
    else:
        req1 = emp1.request()
        req2 = emp2.request()
        resultado = yield env.any_of([req1, req2])
        
        if req1 in resultado:
            req = req1
            req2.cancel()
            emp_elegido = 1
        else:
            req = req2
            req1.cancel()
            emp_elegido = 2

    t_espera = env.now - t_llegada
    stats['espera'].append(t_espera)
    if t_espera > 8:
        stats['espera_mayor_8'] += 1
    
    if emp_elegido == 1:
        t_servicio = tiempo_emp1_A()
        yield env.timeout(t_servicio)
        emp1.release(req)
        stats['uso_emp1'] += t_servicio
    else:
        t_servicio = tiempo_emp2_A()
        yield env.timeout(t_servicio)
        emp2.release(req)
        stats['uso_emp2'] += t_servicio
        
    stats['sistema'].append(t_espera + t_servicio)

def generador_a(env, emp1, emp2, stats, tiempo_limite):
    while env.now < tiempo_limite:
        yield env.timeout(np.random.exponential(5))
        env.process(cliente_escenario_a(env, emp1, emp2, stats))

In [11]:
def cliente_escenario_b(env, kiosco, entrega, tradicional, stats):
    t_llegada = env.now
    
    es_kiosco = np.random.rand() < 0.60
    
    t_espera_total = 0
    t_servicio_total = 0
    
    if es_kiosco:
        t_llegada_q1 = env.now
        req_k = kiosco.request()
        yield req_k
        espera_k = env.now - t_llegada_q1
        
        serv_k = tiempo_kiosco_B()
        yield env.timeout(serv_k)
        kiosco.release(req_k)
        stats['uso_kiosco'] += serv_k
        
        t_llegada_q2 = env.now
        req_e = entrega.request()
        yield req_e
        espera_e = env.now - t_llegada_q2
        
        serv_e = tiempo_entrega_B()
        yield env.timeout(serv_e)
        entrega.release(req_e)
        stats['uso_entrega'] += serv_e
        
        t_espera_total = espera_k + espera_e
        t_servicio_total = serv_k + serv_e
        stats['espera_kiosco'].append(t_espera_total)
        
    else:
        req_t = tradicional.request()
        yield req_t
        t_espera_total = env.now - t_llegada
        
        serv_t = tiempo_tradicional_B()
        yield env.timeout(serv_t)
        tradicional.release(req_t)
        stats['uso_tradicional'] += serv_t
        
        t_servicio_total = serv_t
        stats['espera_tradicional'].append(t_espera_total)

    stats['espera'].append(t_espera_total)
    stats['sistema'].append(t_espera_total + t_servicio_total)
    if t_espera_total > 8:
        stats['espera_mayor_8'] += 1

def generador_b(env, kiosco, entrega, tradicional, stats, tiempo_limite):
    while env.now < tiempo_limite:
        yield env.timeout(np.random.exponential(5))
        env.process(cliente_escenario_b(env, kiosco, entrega, tradicional, stats))


In [12]:
def simular_turnos(escenario, num_turnos, minutos=300):
    res = {
        'espera_gen': [], 'sistema': [], 'mayor_8_pct': [],
        'util_1': [], 'util_2': [], 'util_3': [],
        'espera_k': [], 'espera_t': []
    }
    
    for _ in range(num_turnos):
        env = simpy.Environment()
        if escenario == 'A':
            emp1 = simpy.Resource(env, capacity=1)
            emp2 = simpy.Resource(env, capacity=1)
            stats = {'espera': [], 'sistema': [], 'espera_mayor_8': 0, 'uso_emp1': 0, 'uso_emp2': 0}
            env.process(generador_a(env, emp1, emp2, stats, minutos))
        else:
            kiosco = simpy.Resource(env, capacity=1)
            entrega = simpy.Resource(env, capacity=1)
            trad = simpy.Resource(env, capacity=1)
            stats = {'espera': [], 'sistema': [], 'espera_mayor_8': 0, 
                     'uso_kiosco': 0, 'uso_entrega': 0, 'uso_tradicional': 0,
                     'espera_kiosco': [], 'espera_tradicional': []}
            env.process(generador_b(env, kiosco, entrega, trad, stats, minutos))
            
        env.run()
        
        total_clientes = len(stats['espera'])
        if total_clientes > 0:
            res['espera_gen'].append(np.mean(stats['espera']))
            res['sistema'].append(np.mean(stats['sistema']))
            res['mayor_8_pct'].append((stats['espera_mayor_8'] / total_clientes) * 100)
            
        if escenario == 'A':
            res['util_1'].append((stats['uso_emp1'] / minutos) * 100)
            res['util_2'].append((stats['uso_emp2'] / minutos) * 100)
        else:
            res['util_1'].append((stats['uso_kiosco'] / minutos) * 100)
            res['util_2'].append((stats['uso_entrega'] / minutos) * 100)
            res['util_3'].append((stats['uso_tradicional'] / minutos) * 100)
            if stats['espera_kiosco']: res['espera_k'].append(np.mean(stats['espera_kiosco']))
            if stats['espera_tradicional']: res['espera_t'].append(np.mean(stats['espera_tradicional']))
  
    if escenario == 'A':
        return {
            'Espera General (min)': round(np.mean(res['espera_gen']), 2),
            'Sistema Total (min)': round(np.mean(res['sistema']), 2),
            'Esperan > 8 min (%)': round(np.mean(res['mayor_8_pct']), 2),
            'Utilización Emp 1 (%)': round(np.mean(res['util_1']), 2),
            'Utilización Emp 2 (%)': round(np.mean(res['util_2']), 2),
            'Detalle Espera': 'N/A'
        }
    else:
        esp_k = round(np.mean(res['espera_k']), 2) if res['espera_k'] else 0
        esp_t = round(np.mean(res['espera_t']), 2) if res['espera_t'] else 0
        return {
            'Espera General (min)': round(np.mean(res['espera_gen']), 2),
            'Sistema Total (min)': round(np.mean(res['sistema']), 2),
            'Esperan > 8 min (%)': round(np.mean(res['mayor_8_pct']), 2),
            'Utilización Kiosco (%)': round(np.mean(res['util_1']), 2),
            'Utilización Entrega (%)': round(np.mean(res['util_2']), 2),
            'Util. Tradicional (%)': round(np.mean(res['util_3']), 2),
            'Detalle Espera': f"Kiosco: {esp_k}m | Trad: {esp_t}m"
        }
    
turnos_a_simular = [1, 5, 10, 30]

for turnos in turnos_a_simular:
    print(f"\n{'='*65}")
    print(f"RESULTADOS SIMULANDO {turnos} TURNO(S) DE 300 MINUTOS")
    print(f"{'='*65}")
    
    res_A = simular_turnos('A', turnos)
    res_B = simular_turnos('B', turnos)
    
    df = pd.DataFrame([res_A, res_B], index=['A (Mostrador Único)', 'B (Kiosco + Tradicional)'])
    print(df.fillna('-').to_string())


RESULTADOS SIMULANDO 1 TURNO(S) DE 300 MINUTOS
                          Espera General (min)  Sistema Total (min)  Esperan > 8 min (%) Utilización Emp 1 (%) Utilización Emp 2 (%)               Detalle Espera Utilización Kiosco (%) Utilización Entrega (%) Util. Tradicional (%)
A (Mostrador Único)                       1.13                 6.07                 0.00                 63.94                 52.93                          N/A                      -                       -                     -
B (Kiosco + Tradicional)                  1.92                 8.03                 6.94                     -                     -  Kiosco: 1.97m | Trad: 1.83m                  46.47                   47.37                 52.85

RESULTADOS SIMULANDO 5 TURNO(S) DE 300 MINUTOS
                          Espera General (min)  Sistema Total (min)  Esperan > 8 min (%) Utilización Emp 1 (%) Utilización Emp 2 (%)               Detalle Espera Utilización Kiosco (%) Utilización Entrega (%) Ut

In [14]:
#Problema 3
def t_med1_A():
    t = np.random.normal(8, 2)
    while t <= 0: t = np.random.normal(8, 2)
    return t

def t_med2_A():
    return np.random.uniform(10, 15)

def t_triage_B():
    return np.random.uniform(2, 5)

def t_med1_B():
    t = np.random.normal(6, 1)
    while t <= 0: t = np.random.normal(6, 1)
    return t

def t_med2_B():
    return np.random.uniform(12, 18)

In [15]:
def paciente_escenario_a(env, med1, med2, stats):
    t_llegada = env.now
    
    if med1.count == 0:
        req = med1.request()
        med_elegido = 1
        yield req
    elif med2.count == 0:
        req = med2.request()
        med_elegido = 2
        yield req
    else:
        req1 = med1.request()
        req2 = med2.request()
        resultado = yield env.any_of([req1, req2])
        if req1 in resultado:
            req = req1
            req2.cancel()
            med_elegido = 1
        else:
            req = req2
            req1.cancel()
            med_elegido = 2

    t_espera = env.now - t_llegada
    stats['esperas'].append(t_espera)
    
    if med_elegido == 1:
        t_serv = t_med1_A()
        yield env.timeout(t_serv)
        med1.release(req)
        stats['uso_m1'] += t_serv
    else:
        t_serv = t_med2_A()
        yield env.timeout(t_serv)
        med2.release(req)
        stats['uso_m2'] += t_serv
        
    stats['sistemas'].append(t_espera + t_serv)
    stats['atendidos'] += 1

def generador_a(env, med1, med2, stats, limite):
    while env.now < limite:
        yield env.timeout(np.random.exponential(9))
        env.process(paciente_escenario_a(env, med1, med2, stats))

In [16]:
def paciente_escenario_b(env, triage, med1, med2, stats):
    t_llegada = env.now
    
    req_t = triage.request()
    yield req_t
    espera_triage = env.now - t_llegada
    
    serv_t = t_triage_B()
    yield env.timeout(serv_t)
    triage.release(req_t)
    stats['uso_triage'] += serv_t
    
    es_leve = np.random.rand() < 0.80
    t_llegada_med = env.now
    
    if es_leve:
        req_m = med1.request()
        yield req_m
        espera_med = env.now - t_llegada_med
        
        serv_m = t_med1_B()
        yield env.timeout(serv_m)
        med1.release(req_m)
        
        stats['uso_m1'] += serv_m
        stats['sistema_leve'].append((env.now - t_llegada))
    else:
        req_m = med2.request()
        yield req_m
        espera_med = env.now - t_llegada_med
        
        serv_m = t_med2_B()
        yield env.timeout(serv_m)
        med2.release(req_m)
        
        stats['uso_m2'] += serv_m
        stats['sistema_grave'].append((env.now - t_llegada))
        
    stats['esperas'].append(espera_triage + espera_med)
    stats['atendidos'] += 1

def generador_b(env, triage, med1, med2, stats, limite):
    while env.now < limite:
        yield env.timeout(np.random.exponential(9))
        env.process(paciente_escenario_b(env, triage, med1, med2, stats))

In [17]:
def simular_jornadas(escenario, jornadas, minutos=360):
    res = {
        'espera': [], 'sistema_gral': [], 'atendidos': [],
        'util_m1': [], 'util_m2': [], 'util_triage': [],
        'sistema_leve': [], 'sistema_grave': []
    }
    
    for _ in range(jornadas):
        env = simpy.Environment()
        
        if escenario == 'A':
            med1 = simpy.Resource(env, capacity=1)
            med2 = simpy.Resource(env, capacity=1)
            stats = {'esperas': [], 'sistemas': [], 'atendidos': 0, 'uso_m1': 0, 'uso_m2': 0}
            env.process(generador_a(env, med1, med2, stats, minutos))
        else:
            triage = simpy.Resource(env, capacity=1)
            med1 = simpy.Resource(env, capacity=1) 
            med2 = simpy.Resource(env, capacity=1) 
            stats = {'esperas': [], 'atendidos': 0, 'uso_m1': 0, 'uso_m2': 0, 'uso_triage': 0,
                     'sistema_leve': [], 'sistema_grave': []}
            env.process(generador_b(env, triage, med1, med2, stats, minutos))
            
        env.run()
    
        if stats['esperas']: res['espera'].append(np.mean(stats['esperas']))
        res['atendidos'].append(stats['atendidos'])
        res['util_m1'].append((stats['uso_m1'] / minutos) * 100)
        res['util_m2'].append((stats['uso_m2'] / minutos) * 100)
        
        if escenario == 'A':
            if stats['sistemas']: res['sistema_gral'].append(np.mean(stats['sistemas']))
        else:
            res['util_triage'].append((stats['uso_triage'] / minutos) * 100)
            if stats['sistema_leve']: res['sistema_leve'].append(np.mean(stats['sistema_leve']))
            if stats['sistema_grave']: res['sistema_grave'].append(np.mean(stats['sistema_grave']))

    if escenario == 'A':
        return {
            'Espera Promedio (min)': round(np.mean(res['espera']), 2),
            'Sistema Promedio (min)': round(np.mean(res['sistema_gral']), 2),
            'Pacientes Atendidos/Día': round(np.mean(res['atendidos']), 0),
            'Utilización Med 1 (%)': round(np.mean(res['util_m1']), 2),
            'Utilización Med 2 (%)': round(np.mean(res['util_m2']), 2),
            'Utilización Triage (%)': 'N/A'
        }
    else:
        sis_leve = round(np.mean(res['sistema_leve']), 2) if res['sistema_leve'] else 0
        sis_grave = round(np.mean(res['sistema_grave']), 2) if res['sistema_grave'] else 0
        sis_gral = (sis_leve * 0.8) + (sis_grave * 0.2)
        
        return {
            'Espera Promedio (min)': round(np.mean(res['espera']), 2),
            'Sistema Promedio (min)': f"Leve: {sis_leve} | Grave: {sis_grave}",
            'Pacientes Atendidos/Día': round(np.mean(res['atendidos']), 0),
            'Utilización Med 1 (%)': round(np.mean(res['util_m1']), 2),
            'Utilización Med 2 (%)': round(np.mean(res['util_m2']), 2),
            'Utilización Triage (%)': round(np.mean(res['util_triage']), 2)
        }

jornadas_simular = [1, 5, 10, 30]

for jornadas in jornadas_simular:
    print(f"\n{'='*75}")
    print(f"RESULTADOS SIMULANDO {jornadas} JORNADA(S) DE 360 MINUTOS")
    print(f"{'='*75}")
    
    res_A = simular_jornadas('A', jornadas)
    res_B = simular_jornadas('B', jornadas)
    
    df = pd.DataFrame([res_A, res_B], index=['Escenario A (Directo)', 'Escenario B (Con Triage)'])
    print(df.to_string())


RESULTADOS SIMULANDO 1 JORNADA(S) DE 360 MINUTOS
                          Espera Promedio (min)      Sistema Promedio (min)  Pacientes Atendidos/Día  Utilización Med 1 (%)  Utilización Med 2 (%) Utilización Triage (%)
Escenario A (Directo)                      2.30                       11.34                     38.0                  60.08                  35.26                    N/A
Escenario B (Con Triage)                   4.61  Leve: 12.47 | Grave: 28.31                     33.0                  42.25                  32.29                  32.74

RESULTADOS SIMULANDO 5 JORNADA(S) DE 360 MINUTOS
                          Espera Promedio (min)      Sistema Promedio (min)  Pacientes Atendidos/Día  Utilización Med 1 (%)  Utilización Med 2 (%) Utilización Triage (%)
Escenario A (Directo)                      3.55                       13.14                     45.0                  65.45                  55.54                    N/A
Escenario B (Con Triage)                   3.74  L

In [ ]:
#faltan prob 4 y 5 
#faltan resolver preguntas por cada problema